# LunarSite — Getting Started

**Project:** [LunarSite on GitHub](https://github.com/AlanSEncinas/LunarSite) · [Live demo](https://lunarsite.streamlit.app)

End-to-end ML pipeline for lunar south pole landing site selection. This notebook demonstrates the **Stage 2 terrain segmenter** (production checkpoint and the MC Dropout calibrated-uncertainty variant) on real moon imagery — the fastest way to see what the model does without cloning the repo.

## What you'll see
1. Production U-Net + ResNet-34 segmenter (test mIoU **0.8456** with flip TTA) running on real lunar photographs.
2. MC Dropout variant (val mIoU 0.8134, **ECE 0.0072** across 46M pixels) producing per-pixel epistemic uncertainty — entropy + mutual information. On out-of-distribution real-moon photos, mutual info is **4.7× higher** than on the synthetic training distribution.

Datasets attached: `encinas88/lunarsite-weights` (model checkpoints) and `romainpessia/artificial-lunar-rocky-landscape-dataset` (which also contains the 72 real moon photographs we'll predict on).

In [ ]:
!pip install -q segmentation_models_pytorch

import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Kaggle's dataset mount points vary by version. Discover paths dynamically.
def find_first_with(filename, root='/kaggle/input'):
    """Walk /kaggle/input and return the directory containing `filename`."""
    for dirpath, _dirs, files in os.walk(root):
        if filename in files:
            return Path(dirpath)
    raise FileNotFoundError(f'{filename} not found anywhere under {root}')

WEIGHTS_DIR = find_first_with('best_resnet34.pt')
IMAGES_DIR = find_first_with('g_PCAM1.png').parent  # the parent of /real_moon_images

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device:           {DEVICE}')
print(f'Weights mount:    {WEIGHTS_DIR}')
print(f'Images mount:     {IMAGES_DIR}')
print(f'Weights files:    {sorted(p.name for p in WEIGHTS_DIR.glob("*.pt"))}')
print(f'Real moon images: {len(list((IMAGES_DIR / "real_moon_images").glob("*.png")))} files')

## Stage 2 — Production terrain segmenter

U-Net + ResNet-34 encoder, 4 classes: `background`, `small_rocks`, `large_rocks`, `sky`. Trained exclusively on 9,766 synthetic Unreal Engine lunar scenes; below we demonstrate zero-shot transfer to real moon photographs.

In [ ]:
NC = 4
CLASS_NAMES = ['background', 'small_rocks', 'large_rocks', 'sky']
CLASS_COLORS = np.array(
    [[0, 0, 0], [255, 165, 0], [255, 0, 0], [135, 206, 235]],
    dtype=np.uint8,
)
INPUT_SIZE = 480


def add_mc_dropout(model, p=0.1):
    """Inject Dropout2d after every ReLU (matches the LunarSite training-time setup)."""
    for name, child in model.named_children():
        if isinstance(child, nn.ReLU):
            setattr(model, name, nn.Sequential(nn.ReLU(inplace=True), nn.Dropout2d(p=p)))
        else:
            add_mc_dropout(child, p)


def load_segmenter(weight_path, *, mc_dropout_p=None):
    """Load a 4-class lunar terrain segmenter. Pass mc_dropout_p to inject dropout
    for the MC variant (must match the p the checkpoint was trained with)."""
    model = smp.Unet('resnet34', encoder_weights=None, in_channels=3, classes=NC)
    if mc_dropout_p is not None:
        add_mc_dropout(model, p=mc_dropout_p)
    ckpt = torch.load(weight_path, map_location=DEVICE, weights_only=False)
    state = ckpt.get('model_state_dict') or ckpt.get('model')
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


def preprocess(img_path, size=INPUT_SIZE):
    img = np.array(Image.open(img_path).convert('RGB'))
    h, w = img.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    cropped = img[y0:y0 + side, x0:x0 + side]
    resized = cv2.resize(cropped, (size, size), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(
        resized.transpose(2, 0, 1).astype(np.float32) / 255.0
    ).unsqueeze(0).to(DEVICE)
    return tensor, resized


def colorize(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c in range(NC):
        out[mask == c] = CLASS_COLORS[c]
    return out


segmenter = load_segmenter(WEIGHTS_DIR / 'best_resnet34.pt')
print(f'Loaded production segmenter ({sum(p.numel() for p in segmenter.parameters()):,} params)')

## Flip-TTA inference on real moon photographs

The model was trained only on synthetic data — here it predicts on three real moon photos from the same source dataset. Inference averages predictions across 4 flips (identity + horizontal + vertical + both) for robustness.

In [ ]:
@torch.no_grad()
def predict_with_tta(model, x):
    probs = F.softmax(model(x), dim=1)
    for dims in ([3], [2], [2, 3]):
        probs = probs + torch.flip(F.softmax(model(torch.flip(x, dims=dims)), dim=1), dims=dims)
    return (probs / 4).argmax(1)[0].cpu().numpy().astype(np.uint8)


real_imgs = sorted((IMAGES_DIR / 'real_moon_images').glob('*.png'))[:3]
fig, axes = plt.subplots(len(real_imgs), 2, figsize=(11, 5 * len(real_imgs)))
for i, img_path in enumerate(real_imgs):
    x, rgb = preprocess(img_path)
    mask = predict_with_tta(segmenter, x)
    overlay = (0.5 * rgb + 0.5 * colorize(mask)).astype(np.uint8)
    axes[i, 0].imshow(rgb)
    axes[i, 0].axis('off')
    axes[i, 0].set_title(f'Input: {img_path.name}')
    axes[i, 1].imshow(overlay)
    axes[i, 1].axis('off')
    axes[i, 1].set_title('Prediction overlay  (orange=small rocks, red=large rocks, blue=sky)')
plt.tight_layout()
plt.show()

## MC Dropout — calibrated epistemic uncertainty (Layer 3)

The `best_segmenter_mcdropout.pt` checkpoint was fine-tuned with 27 `Dropout2d(p=0.1)` modules injected after every ReLU. Training-time dropout makes the model's confidence well-calibrated: **across 46 M validation pixels, when the model predicts 99 % confidence it's right 99.3 % of the time (ECE = 0.0072).**

More importantly for landing-site safety: when shown real moon photos the model wasn't trained on, **mutual information jumps 4.7× vs in-domain validation** (0.192 vs 0.041). The model can flag *I don't know* via measurable uncertainty instead of being confidently wrong on out-of-distribution inputs.

In [ ]:
def enable_mc_dropout(model):
    """Switch only the dropout modules back to training mode for MC sampling."""
    for m in model.modules():
        if isinstance(m, nn.Dropout2d):
            m.train()


@torch.no_grad()
def mc_predict(model, x, n_samples=20):
    model.eval()
    enable_mc_dropout(model)
    samples = torch.stack([F.softmax(model(x), dim=1) for _ in range(n_samples)]).squeeze(1)
    mean_probs = samples.mean(0)
    pred = mean_probs.argmax(0).cpu().numpy().astype(np.uint8)
    entropy = -(mean_probs * torch.log(mean_probs + 1e-10)).sum(0)
    per_sample_H = -(samples * torch.log(samples + 1e-10)).sum(1)
    mi = entropy - per_sample_H.mean(0)
    return pred, entropy.cpu().numpy(), mi.cpu().numpy()


mc_segmenter = load_segmenter(
    WEIGHTS_DIR / 'best_segmenter_mcdropout.pt',
    mc_dropout_p=0.1,  # must match training-time p
)
x, rgb = preprocess(real_imgs[0])
pred, entropy, mi = mc_predict(mc_segmenter, x, n_samples=20)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(rgb); axes[0].axis('off'); axes[0].set_title('Input')
axes[1].imshow((0.5 * rgb + 0.5 * colorize(pred)).astype(np.uint8))
axes[1].axis('off'); axes[1].set_title('MC-mean prediction')
axes[2].imshow(entropy, cmap='hot')
axes[2].axis('off'); axes[2].set_title(f'Predictive entropy (total)\nmean = {entropy.mean():.3f}')
axes[3].imshow(mi, cmap='hot')
axes[3].axis('off'); axes[3].set_title(f'Mutual info (epistemic)\nmean = {mi.mean():.3f}')
plt.tight_layout()
plt.show()

print("\nMutual information is the operationally useful number:")
print("  In-domain val (synthetic):  ~0.041 mean MI")
print(f"  Real moon (this image):     {mi.mean():.3f} mean MI")
print("  Higher MI on OOD inputs = the model knows when it's looking at something unfamiliar.")

## Next steps

**The full LunarSite pipeline** does more than terrain segmentation — Stage 1 detects craters on the LOLA south pole DEM, and Stage 3 ranks 315,034 candidate 1 km landing cells across 80°S–90°S with PSR-aware features and SHAP explainability (5/9 NASA Artemis III candidate regions overlap top-1000 cells).

- **Live interactive demo:** [lunarsite.streamlit.app](https://lunarsite.streamlit.app)
- **Source code:** [github.com/AlanSEncinas/LunarSite](https://github.com/AlanSEncinas/LunarSite)
- **Companion training dataset (Stage 1 fine-tune):** [encinas88/lunarsite-southpole-finetune](https://www.kaggle.com/datasets/encinas88/lunarsite-southpole-finetune)
- **Author site:** [alanscottencinas.com](https://alanscottencinas.com)

**Citation:**

```
Encinas, A. (2026). LunarSite — end-to-end ML pipeline for lunar south pole
landing site selection. https://github.com/AlanSEncinas/LunarSite
```